### Investigate the hypothesis that the variants with higher effect sizes are enriched for gwas hits

### Process:
1. read in the metadata file which was used to generate the data for the langenberg group
2. identify which variants are matchable (SPDIs)
- Do we have matching problems still?
3. Get the ALT ids which are associated with a GWAS hit
- Add the has_GWAS_hit column to the variant data
4. Add the is significant column
5. Add the has_high_effect column
6. Find out how to compare the proportion of varaints with GWAS hits in the significant variants vs the non-significant variants
7. Plot the results   

In [41]:
import pandas as pd
import yaml


# load helpful functions
import sys
sys.path.append('../../00_helpful_functions')
import helpful_functions as hf

# config
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/global80K_config.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

In [42]:
col_name = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class' # SNV
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'


import ast

# Function to safely evaluate string representations of lists
def safe_eval(x):
    if pd.isna(x):
        return None
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return x

sig_level = 0.1

In [43]:
#first version
summary_table = "/home/kisa/coding/80K_MPRA/collaborations/langenberg/sig_variants_80k_MPRA_unique_kircher.gwas_summary.tsv"
overlap_table = "/home/kisa/coding/80K_MPRA/collaborations/langenberg/sig_variants_80k_MPRA_unique_kircher.gwas_overlap.tsv"

# all variants
summary_table = "/home/kisa/coding/80K_MPRA/collaborations/langenberg/all_variants_80k_MPRA_unique_kircher.gwas_summary.tsv.gz"
overlap_table = "/home/kisa/coding/80K_MPRA/collaborations/langenberg/all_variants_80k_MPRA_unique_kircher.gwas_overlap.tsv.gz"

summary_df = pd.read_csv(summary_table, sep="\t", low_memory=False)
overlap_df = pd.read_csv(overlap_table, sep="\t", low_memory=False)

In [44]:
summary_df["has_gwas_annotation"] = summary_df["mapped_trait_efo"].notna()
gwas_annotation_spdis = summary_df.loc[summary_df['has_gwas_annotation']]['spdi'].to_list()


In [45]:
metadata_file_path ="/home/kisa/coding/80K_MPRA/element_analysis_output/202502_element_analysis/element_metadata_with_bcalm_2025_02_bbmap_vs_scrambled.tsv.gz"
metadata_file_path ="/home/kisa/coding/80K_MPRA/element_analysis_output/202502_element_analysis/NGN2_element_metadata_with_bcalm_2025_04_normalized_counts_bbmap_vs_scrambled.tsv.gz"
metadata_file = pd.read_csv(metadata_file_path, sep="\t", low_memory=False)

# list columns col_variant_class, col_variant_pos, col_SPDI, col_allele
list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]

# Apply the safe_eval function to the specified columns
for col in list_columns:
    metadata_file[col] = metadata_file[col].apply(safe_eval)


In [46]:
variant_metadata_path ="/home/kisa/coding/80K_MPRA/element_variant_analysis_output/NGN2_variants_metadata_with_bcalm_2025_02_bbmap_scrambled.tsv.gz"
variant_bcalm_df = pd.read_csv(variant_metadata_path, sep="\t", low_memory=False)

# # list columns col_variant_class, col_variant_pos, col_SPDI, col_allele
list_columns = [col_variant_pos]

# Apply the safe_eval function to the specified columns
for col in list_columns:
    variant_bcalm_df[col] = variant_bcalm_df[col].apply(safe_eval)

variant_bcalm_df = variant_bcalm_df.loc[variant_bcalm_df['bcalm_variant_effect_data_exists']].copy()

In [47]:
variant_bcalm_df

,ID,REF,ALT,ref_sequence,alt_sequence,variant_pos,bcalm_variant_effect_data_exists,bcalm_variant_effect_adjusted_p_value,bcalm_variant_effect_log_ratio_activity,bcalm_reference_adjusted_p_value,bcalm_reference_log_ratio_activity,bcalm_alternative_adjusted_p_value,bcalm_alternative_log_ratio_activity,label
3,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCATGCGGTGGCCACAGCCTCGGGTGAGTTCCGGTTCCAAAGTACC...,CCATGCGGTGGCCACAGCCTCGGGTGAGTTCCGGTTCCAAAGTACC...,[116],True,0.979716,0.032099,1.000000,-0.664178,1.000000e+00,-0.675062,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,GGACTCCGGTGCCTTCGCATTCCCGAGCTGTTTTTGCTTCTGGAAG...,GGACTCCGGTGCCTTCGCATTCCCGAGCTGTTTTTGCTTCTGGAAG...,[205],True,0.929889,-0.053121,1.000000,-0.011245,1.000000e+00,-0.028705,cardiac_neuro_cava_random
6,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCTCCACTTGTCAGGAAGCCTGACCCCCAATCCCCTCCCGCCTGAC...,CCTCCACTTGTCAGGAAGCCTGACCCCCAATCCCCTCCCGCCTGAC...,[71],True,0.775543,-0.212077,1.000000,-0.530696,1.000000e+00,-0.725193,cardiac_neuro_cava_random
7,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCTCCACTTGTCAGGAAGCCTGACCCCCAATCCCCTCCCGCCTGAC...,CCTCCACTTGTCAGGAAGCCTGACCCCCAATCCCCTCCCGCCTGAC...,[144],True,0.999288,0.001458,1.000000,-0.530696,1.000000e+00,-0.612857,cardiac_neuro_cava_random
8,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCTCCACTTGTCAGGAAGCCTGACCCCCAATCCCCTCCCGCCTGAC...,CCTCCACTTGTCAGGAAGCCTGACCCCCAATCCCCTCCCGCCTGAC...,[183],True,0.729637,-0.211652,1.000000,-0.530696,8.129874e-01,-0.774325,cardiac_neuro_cava_random
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47039,C_positive_heart_CAD:ALT_rs7865618_rs7865618,C_positive_heart_CAD:REF_rs7865618,C_positive_heart_CAD:ALT_rs7865618_rs7865618,TATTGATAACAGGGGATGGATTCTTGTGGACAAAAAAATTTAGAAT...,TATTGATAACAGGGGATGGATTCTTGTGGACAAAAAAATTTAGAAT...,[136],True,0.101141,0.229206,1.000000,-0.322271,1.000000e+00,-0.106460,C_positive_heart_CAD
47040,C_positive_heart_CAD:ALT_rs4977757_rs4977757,C_positive_heart_CAD:REF_rs4977757,C_positive_heart_CAD:ALT_rs4977757_rs4977757,CTGATGGGCTTCCCTTTGTGGGTAACCCGACCTTTCTCTCTGGCTG...,CTGATGGGCTTCCCTTTGTGGGTAACCCGACCTTTCTCTCTGGCTG...,[136],True,0.222166,-0.468693,0.000576,-1.130559,1.863777e-11,-1.557981,C_positive_heart_CAD
47041,C_positive_heart_CAD:ALT_rs1537373_rs1537373,C_positive_heart_CAD:REF_rs1537373,C_positive_heart_CAD:ALT_rs1537373_rs1537373,AGAAAACCATACCCACTTTCCCACATATCCCAACTATGACTGGGCA...,AGAAAACCATACCCACTTTCCCACATATCCCAACTATGACTGGGCA...,[136],True,0.518546,0.246189,1.000000,-0.240604,1.000000e+00,0.009369,C_positive_heart_CAD
47042,C_positive_heart_CAD:ALT_rs10811656_rs10811656,C_positive_heart_CAD:REF_rs10811656,C_positive_heart_CAD:ALT_rs10811656_rs10811656,AAATTAAAAGCTTCTAAACTAACAAACAGCCAATTTGTGGAGTGTC...,AAATTAAAAGCTTCTAAACTAACAAACAGCCAATTTGTGGAGTGTC...,[136],True,0.812184,0.128220,0.576842,-0.758037,1.000000e+00,-0.630269,C_positive_heart_CAD


In [48]:
# read variant table and add the spdi column there as well
metadata_file_tested_alt = metadata_file.loc[metadata_file["name"].str.startswith("cardiac_neuro_cava_random:ALT_") & (metadata_file[col_SPDI].notna())].copy()

In [49]:
metadata_file_tested_alt # 46,374

,name,sequence,category,class,source,ref,chr,start,end,strand,...,SPDI,allele,info,bcalm_element_adjusted_p_value,bcalm_element_log_ratio_activity,bcalm_data_exists,label,not_shifted_bcalm_element_log_ratio_activity,bcalm_element_log_ratio_activity_z_score_neg_ctrl,bcalm_element_log_ratio_activity_z_score_scrambled_ctrl
29853,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,GTCCCAGCTCCCCACTGATGTGAAAGGTGGTGGTGAGTTAACAGCT...,variant,test,NaN,GRCh38,chr1,2179507.0,2179777.0,+,...,[NC_000001.11:2179590:T:C],[alt],NaN,5.965362e-07,0.190591,True,cardiac_neuro_cava_random,0.190591,7.760601,5.320544
29854,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCTGATCTGCCCTGTCCGTGACGCTTCTGCTCAGTAGCTGAGCACG...,variant,test,NaN,GRCh38,chr1,2191262.0,2191532.0,+,...,[NC_000001.11:2191443:G:A],[alt],NaN,NaN,NaN,False,cardiac_neuro_cava_random,NaN,NaN,NaN
29855,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCTCTGGGTGACCCGGAGAACACCAAGGCTGTGAGAAATGGGATGC...,variant,test,NaN,GRCh38,chr1,2191971.0,2192241.0,+,...,[NC_000001.11:2192014:G:T],[alt],NaN,NaN,NaN,False,cardiac_neuro_cava_random,NaN,NaN,NaN
29856,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCATGCGGTGGCCACAGCCTCGGGTGAGTTCCGGTTCCAAAGTACC...,variant,test,NaN,GRCh38,chr1,2192249.0,2192519.0,+,...,[NC_000001.11:2192365:T:G],[alt],NaN,1.000000e+00,-0.072051,True,cardiac_neuro_cava_random,-0.072051,0.324578,-1.471449
29857,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,GGACTCCGGTGCCTTCGCATTCCCGAGCTGTTTTTGCTTCTGGAAG...,variant,test,NaN,GRCh38,chr1,2192936.0,2193206.0,+,...,[NC_000001.11:2193141:G:A],[alt],NaN,1.000000e+00,-0.005889,True,cardiac_neuro_cava_random,-0.005889,2.197787,0.239523
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76222,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,GGAGCTCTGCCTCACCCCACCTGGCCCCAATTGTCCAGCTTGTAGA...,variant,test,NaN,GRCh38,chrX,154545000.0,154545270.0,-,...,[NC_000023.11:154545205:A:G],[alt],NaN,4.511542e-01,0.065547,True,cardiac_neuro_cava_random,0.065547,4.220287,2.086855
76223,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,ATGTCTGAATTCACCTCCAAATAATGGGAAAACTCCTAGGTATATA...,variant,test,NaN,GRCh38,chrX,154549802.0,154550072.0,-,...,[NC_000023.11:154549922:T:G],[alt],NaN,9.174673e-02,-0.090558,True,cardiac_neuro_cava_random,-0.090558,-0.199415,-1.950059
76224,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,CCTCTGCCCTCCCTGGCTTCTTCCCCTGTCCCTCCTTTCCCTTCCC...,variant,test,NaN,GRCh38,chrX,154552145.0,154552415.0,-,...,[NC_000023.11:154552288:C:T],[alt],NaN,4.362900e-01,0.052674,True,cardiac_neuro_cava_random,0.052674,3.855845,1.753977
76225,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,CCTCTGCCCTCCCTGGCTTCTTCCCCTGTCCCTCCTTTCCCTTCTC...,variant,test,NaN,GRCh38,chrX,154552145.0,154552415.0,-,...,[NC_000023.11:154552370:G:A],[alt],NaN,1.000000e+00,0.016824,True,cardiac_neuro_cava_random,0.016824,2.840822,0.826865


In [50]:
metadata_file_tested_alt["SPDI"] = metadata_file_tested_alt[col_SPDI].apply(lambda spdi_list: spdi_list[0] if isinstance(spdi_list, list) and len(spdi_list) > 0 else None)

In [51]:
metadata_file_tested_alt['has_gwas_annotation'] = metadata_file_tested_alt["SPDI"].isin(gwas_annotation_spdis)

In [52]:
metadata_file_tested_alt['has_gwas_annotation'].sum()


808

In [54]:
alternative_name_has_gwas_annotation = metadata_file_tested_alt.loc[metadata_file_tested_alt['has_gwas_annotation'], 'name'].to_list()
alternative_name_has_gwas_annotation

['cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778534|1-2210427-C-T',
 'cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778534|1-2210546-C-A',
 'cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778579_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778579|1-2233496-G-T',
 'cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778605_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778605|1-2249085-G-A',
 'cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778610_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778610|1-2250903-T-C',
 'cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E1311725_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E1311725|1-2271528-T-C',
 'cardiac_neuro_cava_random:ALT_PRDM16|ENSG00000142611.17|EH38E2779527_fwd_tile1-1_PRDM16|ENSG00000142611.17|EH38E2779527|1-3033770-G-A',
 'cardiac_neuro_cava_random:ALT_PRDM16|ENSG00000142611.17|EH38E2779663

In [55]:
# add the gwas information to the variant table
variant_bcalm_df['has_gwas_annotation'] = variant_bcalm_df["ALT"].isin(alternative_name_has_gwas_annotation)

In [56]:
variant_bcalm_df['has_gwas_annotation'].sum() # 774


663

In [64]:
variant_bcalm_df['high_variant_effect'] = variant_bcalm_df['bcalm_variant_effect_log_ratio_activity'] > 1

In [57]:
variant_bcalm_df['is_significant'] = variant_bcalm_df['bcalm_variant_effect_adjusted_p_value'] < sig_level

In [58]:
variant_bcalm_df

,ID,REF,ALT,ref_sequence,alt_sequence,variant_pos,bcalm_variant_effect_data_exists,bcalm_variant_effect_adjusted_p_value,bcalm_variant_effect_log_ratio_activity,bcalm_reference_adjusted_p_value,bcalm_reference_log_ratio_activity,bcalm_alternative_adjusted_p_value,bcalm_alternative_log_ratio_activity,label,has_gwas_annotation,is_significant
3,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCATGCGGTGGCCACAGCCTCGGGTGAGTTCCGGTTCCAAAGTACC...,CCATGCGGTGGCCACAGCCTCGGGTGAGTTCCGGTTCCAAAGTACC...,[116],True,0.979716,0.032099,1.000000,-0.664178,1.000000e+00,-0.675062,cardiac_neuro_cava_random,False,False
4,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,GGACTCCGGTGCCTTCGCATTCCCGAGCTGTTTTTGCTTCTGGAAG...,GGACTCCGGTGCCTTCGCATTCCCGAGCTGTTTTTGCTTCTGGAAG...,[205],True,0.929889,-0.053121,1.000000,-0.011245,1.000000e+00,-0.028705,cardiac_neuro_cava_random,False,False
6,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCTCCACTTGTCAGGAAGCCTGACCCCCAATCCCCTCCCGCCTGAC...,CCTCCACTTGTCAGGAAGCCTGACCCCCAATCCCCTCCCGCCTGAC...,[71],True,0.775543,-0.212077,1.000000,-0.530696,1.000000e+00,-0.725193,cardiac_neuro_cava_random,False,False
7,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCTCCACTTGTCAGGAAGCCTGACCCCCAATCCCCTCCCGCCTGAC...,CCTCCACTTGTCAGGAAGCCTGACCCCCAATCCCCTCCCGCCTGAC...,[144],True,0.999288,0.001458,1.000000,-0.530696,1.000000e+00,-0.612857,cardiac_neuro_cava_random,False,False
8,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCTCCACTTGTCAGGAAGCCTGACCCCCAATCCCCTCCCGCCTGAC...,CCTCCACTTGTCAGGAAGCCTGACCCCCAATCCCCTCCCGCCTGAC...,[183],True,0.729637,-0.211652,1.000000,-0.530696,8.129874e-01,-0.774325,cardiac_neuro_cava_random,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47039,C_positive_heart_CAD:ALT_rs7865618_rs7865618,C_positive_heart_CAD:REF_rs7865618,C_positive_heart_CAD:ALT_rs7865618_rs7865618,TATTGATAACAGGGGATGGATTCTTGTGGACAAAAAAATTTAGAAT...,TATTGATAACAGGGGATGGATTCTTGTGGACAAAAAAATTTAGAAT...,[136],True,0.101141,0.229206,1.000000,-0.322271,1.000000e+00,-0.106460,C_positive_heart_CAD,False,False
47040,C_positive_heart_CAD:ALT_rs4977757_rs4977757,C_positive_heart_CAD:REF_rs4977757,C_positive_heart_CAD:ALT_rs4977757_rs4977757,CTGATGGGCTTCCCTTTGTGGGTAACCCGACCTTTCTCTCTGGCTG...,CTGATGGGCTTCCCTTTGTGGGTAACCCGACCTTTCTCTCTGGCTG...,[136],True,0.222166,-0.468693,0.000576,-1.130559,1.863777e-11,-1.557981,C_positive_heart_CAD,False,False
47041,C_positive_heart_CAD:ALT_rs1537373_rs1537373,C_positive_heart_CAD:REF_rs1537373,C_positive_heart_CAD:ALT_rs1537373_rs1537373,AGAAAACCATACCCACTTTCCCACATATCCCAACTATGACTGGGCA...,AGAAAACCATACCCACTTTCCCACATATCCCAACTATGACTGGGCA...,[136],True,0.518546,0.246189,1.000000,-0.240604,1.000000e+00,0.009369,C_positive_heart_CAD,False,False
47042,C_positive_heart_CAD:ALT_rs10811656_rs10811656,C_positive_heart_CAD:REF_rs10811656,C_positive_heart_CAD:ALT_rs10811656_rs10811656,AAATTAAAAGCTTCTAAACTAACAAACAGCCAATTTGTGGAGTGTC...,AAATTAAAAGCTTCTAAACTAACAAACAGCCAATTTGTGGAGTGTC...,[136],True,0.812184,0.128220,0.576842,-0.758037,1.000000e+00,-0.630269,C_positive_heart_CAD,False,False


7. How to compare the proportion of variants with GWAS hits in the significant variants vs the non-significant variants
- Use a chi-squared test to compare the proportions of variants with GWAS hits in the significant and non-significant variants


In [61]:
# test for significant difference in the number of variants with gwas annotation

# Create a contingency table
contingency_table = pd.crosstab(variant_bcalm_df['has_gwas_annotation'], variant_bcalm_df['is_significant'])
contingency_table
# # Perform the Chi-squared test
# from scipy.stats import chi2_contingency

# chi2, p_value, _, _ = chi2_contingency(contingency_table)
# print(f"Chi-squared test statistic: {chi2}")
# print(f"P-value: {p_value}")
# # Interpret the result
# if p_value < 0.05:
#     print("There is a significant association between the two categorical variables.")

is_significant,False,True
has_gwas_annotation,,
False,36731,892
True,651,12


In [ ]:
# compute the proportions:
contingency_table_proportions = contingency_table.div(contingency_table.sum(axis=1), axis=0)
contingency_table_proportions # => no enrichment based on significance

is_significant,False,True
has_gwas_annotation,,
False,0.976291,0.023709
True,0.981900,0.018100


Using high effect not only significants

In [66]:
variant_bcalm_df['high_variant_effect'].sum()

31

In [ ]:

contingency_table = pd.crosstab(variant_bcalm_df['has_gwas_annotation'], variant_bcalm_df['high_variant_effect'])
contingency_table
# no enrichment for high variant effects at all

high_variant_effect,False,True
has_gwas_annotation,,
False,37592,31
True,663,0


### Use only common variants